# Dev Notebook: Plotting of Stability Bahavior

## Imports and Definitions

In [ ]:
# %matplotlib inline
# import ipympl

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps as cm

parent = os.path.abspath("/Users/maxikoehler/Documents/GitHub/diffpssi-ma-kohler/")
sys.path.insert(1, parent)
sys.path.append(
    str(os.path.dirname(os.path.dirname(os.path.abspath("v_stabiilty_plotting.ipynb"))))
)
save = os.path.join(os.getcwd(), "plots/")

import matplotlib as mpl
from tools import *

In [ ]:
# import load models
import development_files.examples.ibb_transformer.ibb_trans_model as ibb
from development_files.examples.simple_load.simple_load_mod import load as simple_load


from src.diffpssi import PowerSystemSimulation as Pss
from src.diffpssi import Recorder

# from src.diffpssi.stability_lib import VoltageStability
from src.diffpssi.stability_lib.voltage import NoseCurve

## 1 Enabling Simulation Recorder Variables 

In [ ]:
def record_dict_smload(simulation, call=False):
    record_dict = {
        "B0: V": simulation.busses[0].get_value("voltage_mag"),
        "B1: V": simulation.busses[1].get_value("voltage_mag"),
        "B0: S": np.abs(simulation.busses[0].get_value("S")),
        "B1: S": np.abs(
            simulation.busses[1].get_value("S")
        ),  # models[0].i_inj / simulation.base_voltage),
        # 'Load 0: current injection':    np.abs(simulation.busses[1].models[0].get_current_injections()),
    }
    if call:
        return record_dict.values()
    else:
        return record_dict

In [ ]:
param_dict_oltc = {
    "t_1": 5,
    "db": 0.05,
    "delta_m": 0.02,
    "m_max": 1.1,
    "m_min": 0.9,
    "v_ref": 1,
}

sim = Pss(
    parallel_sims=1,
    sim_time=40,
    time_step=0.001,
    solver="heun",
    grid_data=simple_load(),
)

sim.add_param_event(1, sim.busses[1].models[0], "p_soll_mw", 800)

rec = Recorder(sim=sim, recorder_dict=record_dict_smload)

sim.set_record_function(rec.record_fun)
record_list = rec.record_list()

t, recorder = sim.run()

In [ ]:
sim_results = pd.DataFrame(recorder[0, :, :], columns=record_list)
sim_results["time"] = np.round(t, 3)
sim_results.set_index("time", inplace=True)

sim_results.head()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(t, np.real(recorder[0, :, 2]), label=record_list[2])
plt.grid()
plt.xlabel("Time [s]")
plt.ylabel("Power in [MVA]")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(t, np.abs(recorder[0, :, 3]), label=record_list[3])
plt.grid()
plt.xlabel("Time [s]")
plt.ylabel("Power in [MVA]")
plt.legend()

plt.show()

## 2 Nose Curve of the System

In [ ]:
p_load = np.linspace(0, 10000, 1000)
tan_phi = np.linspace(-0.3, 0.3, 3)

load_dict = {
    "p": p_load,
    "tan_phi": tan_phi,
}
sim.verbose = False

nc = NoseCurve(load_model=simple_load, loading=load_dict, ps_sim=sim)

res_nc = nc.run_calculation(["B1"])["B1"]

In [ ]:
cmap = plt.get_cmap("bone")

red = 1

p = np.real(sim_results.loc[sim_results.index % red == 0]["B1: S"])
v = sim_results.loc[sim_results.index % red == 0]["B1: V"]

ax = nc.plot_nose_curve(["B1"], size=(9, 5))

load_model = {
    "z": 1,
    "i": 0,
    "p": 0,
}

ax = nc.add_load_to_plot(
    load=[2000, 0], load_model=load_model, bus="B1", current_plot=ax, y_lims=[0.4, 1.2]
)
ax = nc.add_load_to_plot(
    load=[400, 0], load_model=load_model, bus="B1", current_plot=ax, y_lims=[0.4, 1.2]
)

ax.scatter(
    p,
    v,
    marker=".",
    c=sim_results.loc[sim_results.index % red == 0].index,
    cmap=cmap,
    label="Time Development Simulation",
)
ax.legend()

plt.show()

## 3 Nose Curves with OLTC Tap Positions

Using the same simulation object as before, it has got already an OLTC integrated.

In [ ]:
p_load = np.linspace(0, 10000, 1000)
tan_phi = [0]

load_dict = {
    "p": p_load,
    "tan_phi": tan_phi,
}
sim.verbose = False

nc = NoseCurve(load_model=simple_load, loading=load_dict, ps_sim=sim)

res_nc = nc.run_calculation(["B1"])["B1"]

ax_parameter_variation = nc.plot_nose_curve(["B1"], size=(9, 5))
# plt.show()

Define Callable in order to access the changable simulation parameter and run the variation.

In [ ]:
def resetter(sim, value):
    sim.trafos[0].u_l = value * np.ones((1, 1), dtype=float)


res_variation = nc.run_variation_calculation(
    bus="B1", variation_parameter=resetter, variation_values=np.arange(0.9, 1.1, 0.02)
)

In [ ]:
# ax_parameter_variation = nc.plot_nose_curve_variation(param_var_dict=res_variation, current_plot=ax_parameter_variation)
# ax_parameter_variation.plot(res_variation['1.0']['p'], np.abs(res_variation['1.0']['v']), color=ees_blue, label='Normal Configuration')

# for ratio in res_variation.keys():
#     ax_parameter_variation.plot(res_variation[ratio]['p'], np.abs(res_variation[ratio]['v']), label=f'{ratio}')

# ax_parameter_variation.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

oltc_ratios = []

# load_inter = nc.add_load_to_plot(load=[2000,0], load_model=load_model, bus='B1', current_plot=ax, plot_args={'color': ees_green, 'label': r'Z-Load at $P=400$ MW'})

for key in [*res_variation]:
    if key != "1.0":
        (oltc,) = ax.plot(
            res_variation[key]["p"],
            np.abs(res_variation[key]["v"]),
            "--",
            color=ees_red,
        )  # , label='OLTC Dependent Ratio'
        oltc_ratios.append(oltc)
    elif key == "1.0":
        (normal,) = ax.plot(
            res_variation[key]["p"], np.abs(res_variation[key]["v"]), color=ees_blue
        )  #  label=r'Ratio $\vartheta=$'+f'{key}',

ax.grid()
ax.legend(
    handles=[oltc_ratios[0], normal],
    labels=[
        r"OLTC Ratios $\vartheta \in \boldsymbol{R} \setminus \{1.0\}$",
        r"Ratio $\vartheta=1.0$",
    ],
)
ax.set_ylabel("Voltage Magnitude in p.u.")
ax.set_xlabel("Power in MW")

plt.savefig("./plots/nose_curves_with-oltc.pdf")

plt.show()